### Trait prediction from social-niche embeddings

1. **Step 1 - leave-one-phylum-out AUC.** A random forest trained on Traitar
   labels is tested on BacDive labels of a held-out phylum; the focal phylum is
   never in the training set. This is the only source of the AUC numbers
   (paper Fig. 2C) and is *not* used to make predictions.
2. **Step 2 - the trait table.** One forest per trait, fitted on *all* Traitar
   labels, applied to every OTU. OTUs that already carry a Traitar label keep
   it (`source = Traitar`); the rest get the inferred value
   (`source = SNE_predicted`). Each row also carries the step-1 AUC of its own
   phylum, so a prediction never appears without its error bar.

Phylum hold-out is an evaluation scheme, not a deployment scheme: for step 2 it
would only throw away labels. One model per trait, fitted on everything.

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [9]:
# ---- inputs (relative to this notebook) and outputs ----------------------- #
EMB_FILE    = "social_niche_embedding_100.txt"
TAXMAP      = "taxmap_slv_ssu_ref_nr_138.2.txt"
TRAITOR_CSV = "trait_predcit.csv"
BACDIVE_CSV = "bacDive.csv"
AGG_BAC_CSV = "agg_bac.csv"        # BacDive column -> metabolic trait name

AUC_CSV   = "../data/traits_predict/auc_res.csv"       # step 1
TABLE_CSV = "../data/traits_predict/trait_table.csv"   # step 2
Path(AUC_CSV).parent.mkdir(parents=True, exist_ok=True)

In [10]:
BIG4 = ["Bacillota", "Bacteroidota", "Actinomycetota", "Pseudomonadota"]
BIG3 = BIG4[:3]

# Traitar spells it "D-Sorbitol", BacDive "sorbitol"; harmonised to "Sorbitol".
METABOLIC = ["Lactose", "Salicin", "Glycerol", "Melibiose",
             "Maltose", "Sucrose", "Trehalose", "Sorbitol"]

# trait -> (phyla held out one at a time,
#           min BacDive samples per class, min classes, both in the test set)
TRAITS = {
    "Oxygen_Preference": (BIG4, 4, 3),
    "Gram_Status":       (BIG3, 6, 2),
    "Motility":          (BIG3, 6, 2),
    "Spore_Formation":   (BIG4, 6, 2),
    **{m: (BIG3, 4, 2) for m in METABOLIC},
}


def forest():
    """A fresh classifier per fit - nothing is shared between traits."""
    return RandomForestClassifier(n_estimators=1000, random_state=0, n_jobs=-1,
                                  class_weight="balanced")

In [11]:
# --------------------------------------------------------------------------- #
# loading
# --------------------------------------------------------------------------- #
def load_embedding():
    emb = pd.read_csv(EMB_FILE, header=None, sep=" ", low_memory=False, index_col=0)
    return emb.drop(index="<unk>", errors="ignore")


def load_taxonomy():
    """SILVA ranks indexed by "<accession>.<start>.<stop>", i.e. by OTU id."""
    tax = pd.read_csv(TAXMAP, sep="\t", low_memory=False)
    ranks = tax["path"].str.split(";", expand=True).iloc[:, :7]
    ranks.columns = ["k", "p", "c", "o", "f", "g", "s"]
    ranks.index = (tax.iloc[:, 0].astype(str) + "." +
                   tax.iloc[:, 1].astype(str) + "." +
                   tax.iloc[:, 2].astype(str)).values
    return ranks


def combine_labels(df, mapping, exclusive=False):
    """Collapse indicator columns into one categorical column.

    mapping: {column: label}, applied in order (first match wins).
    exclusive=True -> NaN unless exactly one indicator is set.
    """
    out = pd.Series(np.nan, index=df.index, dtype=object)
    for col, label in mapping.items():
        out[out.isna() & (df[col] == 1)] = label
    if exclusive:
        out[df[list(mapping)].sum(axis=1) != 1] = np.nan
    return out


def load_traitor():
    """Traitar predictions on the mapped genomes - the training labels."""
    tr = (pd.read_csv(TRAITOR_CSV, index_col=0).astype(int).replace(3, 1)
            .rename(columns={"D-Sorbitol": "Sorbitol"}))
    out = pd.DataFrame({
        "Oxygen_Preference": combine_labels(
            tr, {"Aerobe": "aerobic", "Facultative": "facultatively",
                 "Anaerobe": "anaerobic"}, exclusive=True),
        "Gram_Status": combine_labels(
            tr, {"Gram negative": "negative", "Gram positive": "positive"},
            exclusive=True),
        "Motility": tr["Motile"],
        "Spore_Formation": tr["Spore formation"],
    })
    return out.join(tr[METABOLIC])


def load_bacdive(index):
    """BacDive measurements - the test labels - re-indexed onto OTU ids.

    BacDive is keyed by bare accession, `index` by "<accession>.<start>.<stop>".
    """
    agg = pd.read_csv(AGG_BAC_CSV)
    agg["level_3"] = agg["level_3"].str.capitalize()
    agg = agg[agg["level_3"].isin(METABOLIC) &
              (agg["level_2"] == "builds_acid_from") & (agg["type"] == 1)]
    sugars = dict(zip(agg["terms"], agg["level_3"]))

    tr = (pd.read_csv(BACDIVE_CSV, low_memory=False)
            .drop_duplicates(subset="16s_ID")
            .set_index("16s_ID")
            .replace({"NA": np.nan, "": np.nan, "-": "no", "+": "yes",
                      "+;NA": np.nan, "mixed": np.nan, "variable": np.nan,
                      "no;yes": np.nan, "negative;positive": np.nan,
                      "negative;variable": np.nan}))
    out = pd.DataFrame({
        "Oxygen_Preference": combine_labels(
            tr, {"aerobe": "aerobic", "facultative.anaerobe": "facultatively",
                 "anaerobe": "anaerobic"}),
        "Gram_Status": tr["gram_stain"],
        "Motility": tr["motility"],
        "Spore_Formation": tr["spore_formation"],
    }).join(tr[list(sugars)].rename(columns=sugars)).replace({"yes": 1, "no": 0})

    acc = pd.Index(index).str.split(".").str[0]
    keep = acc.isin(out.index)
    out = out.loc[acc[keep]]
    out.index = pd.Index(index)[keep]
    return out


def restrict(df, index):
    """Rows of df that exist in index. Unlike reindex this adds no NaN, so a
    0/1 trait column stays integer instead of turning into 0.0/1.0."""
    return df.loc[df.index.intersection(index)]

## Step 1 - leave-one-phylum-out AUC

Train on Traitar, test on BacDive, holding out the focal phylum *and* every
phylum outside the focal set.

In [14]:
def phylum_auc(emb, tax, traitor, bacdive, trait, phylum):
    """AUC for one trait with `phylum` and all non-focal phyla held out.

    Returns None when the resulting test set is too small to score.
    """
    focal, min_count, min_classes = TRAITS[trait]

    test_lab = bacdive[trait].dropna()
    tax_test = tax.loc[test_lab.index]
    held_out = {p for p in tax_test["p"].unique() if p not in focal} | {phylum}
    test_id = tax_test.index[tax_test["p"].isin(held_out)]

    train_lab = traitor[trait].dropna()
    tax_train = tax.loc[train_lab.index]
    train_id = tax_train.index[~tax_train["p"].isin(held_out)]

    y_test = test_lab.loc[test_id].values
    classes, counts = np.unique(y_test, return_counts=True)
    if len(classes) < min_classes or counts.min() < min_count:
        return None

    model = forest().fit(emb.loc[train_id], train_lab.loc[train_id].values)
    if not set(classes).issubset(model.classes_):
        return None                              # a test label never seen in training

    proba = model.predict_proba(emb.loc[test_id])
    if len(model.classes_) > 2:
        return roc_auc_score(y_test, proba, multi_class="ovr", average="macro",
                             labels=model.classes_)
    return roc_auc_score((y_test == model.classes_[1]).astype(int), proba[:, 1])


def leave_one_phylum_out(emb, tax, traitor, bacdive):
    """One row per (trait, held-out phylum) that yields a scorable test set."""
    rows = []
    for trait, (focal, _, _) in TRAITS.items():
        for phylum in focal:
            auc = phylum_auc(emb, tax, traitor, bacdive, trait, phylum)
            if auc is not None:
                rows.append({"traits_type": trait, "tax": phylum, "auc": auc})
    return pd.DataFrame(rows)

In [21]:
table = trait_table(emb, tax, traitor, pd.read_csv(AUC_CSV))
table.to_csv(TABLE_CSV, index=False)
table.head()

,otu_id,phylum,genus,Oxygen_Preference,Oxygen_Preference_prob,Oxygen_Preference_source,Oxygen_Preference_auc,Gram_Status,Gram_Status_prob,Gram_Status_source,...,Sucrose_source,Sucrose_auc,Trehalose,Trehalose_prob,Trehalose_source,Trehalose_auc,Sorbitol,Sorbitol_prob,Sorbitol_source,Sorbitol_auc
0,AAAA02020714.1.1202,Pseudomonadota,Sphingomonas,aerobic,0.538,SNE_predicted,0.868794,negative,0.607,SNE_predicted,...,SNE_predicted,0.578489,0,0.531,SNE_predicted,0.527633,0,0.955,SNE_predicted,0.709677
1,AAFJ01000001.39328.40836,Campylobacterota,Campylobacter,anaerobic,0.676,SNE_predicted,0.839734,negative,0.520,SNE_predicted,...,SNE_predicted,0.578489,0,0.709,SNE_predicted,0.527633,0,0.934,SNE_predicted,0.709677
2,AAQK01001555.694.2198,Bacillota,Solobacterium,anaerobic,0.834,SNE_predicted,0.787446,positive,NaN,Traitar,...,Traitar,NaN,0,NaN,Traitar,NaN,0,NaN,Traitar,NaN
3,AAQK01003909.1492.2988,Actinomycetota,Tractidigestivibacter,anaerobic,NaN,Traitar,NaN,positive,NaN,Traitar,...,Traitar,NaN,1,NaN,Traitar,NaN,0,NaN,Traitar,NaN
4,AATC01000018.2.1513,Bacillota,Incertae Sedis,anaerobic,0.630,SNE_predicted,0.787446,positive,0.636,SNE_predicted,...,SNE_predicted,0.587258,0,0.624,SNE_predicted,0.553865,0,0.953,SNE_predicted,0.709677


In [22]:
# sanity: every OTU covered, and Traitar rows carry their own label unchanged.
# Everything is compared by OTU id - the table is in embedding order, the labels
# in Traitar-file order, so a positional comparison would be meaningless.
assert len(table) == len(emb) and table["otu_id"].is_unique
by_id = table.set_index("otu_id")
for trait in TRAITS:
    known = traitor[trait].dropna()
    assert (by_id[f"{trait}_source"] == "Traitar").sum() == len(known)
    assert (by_id.loc[known.index, f"{trait}_source"] == "Traitar").all()
    assert (by_id.loc[known.index, trait] == known).all()
    # prob and auc belong to inferred rows only
    assert by_id.loc[known.index, [f"{trait}_prob", f"{trait}_auc"]].isna().all().all()
    inferred = by_id.index.difference(known.index)
    assert by_id.loc[inferred, f"{trait}_prob"].notna().all()

table.filter(like="_source").apply(pd.Series.value_counts).T

,SNE_predicted,Traitar
Oxygen_Preference_source,13125,968
Gram_Status_source,12989,1104
Motility_source,12981,1112
Spore_Formation_source,12981,1112
Lactose_source,12981,1112
Salicin_source,12981,1112
Glycerol_source,12981,1112
Melibiose_source,12981,1112
Maltose_source,12981,1112
Sucrose_source,12981,1112


## Step 2 - the trait table

`{trait}_prob` is the forest's probability *for the value in that row*; on a
Traitar row that is in-sample and close to 1, so `source`, not `prob`, is what
tells you whether a value was measured or inferred. `{trait}_auc` is the
step-1 AUC of that OTU's own phylum, falling back to the trait's mean AUC for
phyla outside the focal set.

In [20]:
def trait_table(emb, tax, traitor, auc):
    """One row per OTU: value, probability, source and AUC for every trait."""
    by_phylum = auc.set_index(["traits_type", "tax"])["auc"]
    by_trait = auc.groupby("traits_type")["auc"].mean()

    table = pd.DataFrame({"otu_id": emb.index, "phylum": tax["p"].values,
                          "genus": tax["g"].values})
    for trait in TRAITS:
        known = traitor[trait].dropna()
        labelled = emb.index.isin(known.index)

        model = forest().fit(emb.loc[known.index], known.values)
        proba = model.predict_proba(emb)
        col = {c: i for i, c in enumerate(model.classes_)}

        # start from the prediction, then overwrite the OTUs Traitar already knows
        value = pd.Series(model.classes_[proba.argmax(1)], index=emb.index)
        value[known.index] = known

        # prob and auc qualify an inference. A labelled OTU sat in the training
        # set, so its probability would be in-sample - inflated by construction
        # and evidence of nothing - and the cross-phylum AUC would describe a
        # prediction it never made. Both are left empty on those rows.
        prob = np.array([proba[i, col[v]] for i, v in enumerate(value)])
        auc_of_phylum = np.array([by_phylum.get((trait, p), by_trait.get(trait, np.nan))
                                  for p in table["phylum"]])

        table[trait] = value.values
        table[f"{trait}_prob"] = np.where(labelled, np.nan, prob)
        table[f"{trait}_source"] = np.where(labelled, "Traitar", "SNE_predicted")
        table[f"{trait}_auc"] = np.where(labelled, np.nan, auc_of_phylum)
    return table

In [19]:
# sanity: every OTU covered, and Traitar rows carry their own label unchanged.
# Everything is compared by OTU id - the table is in embedding order, the labels
# in Traitar-file order, so a positional comparison would be meaningless.
assert len(table) == len(emb) and table["otu_id"].is_unique
by_id = table.set_index("otu_id")
for trait in TRAITS:
    known = traitor[trait].dropna()
    assert (by_id[f"{trait}_source"] == "Traitar").sum() == len(known)
    assert (by_id.loc[known.index, f"{trait}_source"] == "Traitar").all()
    assert (by_id.loc[known.index, trait] == known).all()

table.filter(like="_source").apply(pd.Series.value_counts).T

,SNE_predicted,Traitar
Oxygen_Preference_source,13125,968
Gram_Status_source,12989,1104
Motility_source,12981,1112
Spore_Formation_source,12981,1112
Lactose_source,12981,1112
Salicin_source,12981,1112
Glycerol_source,12981,1112
Melibiose_source,12981,1112
Maltose_source,12981,1112
Sucrose_source,12981,1112
